# Label shards (GPU)

Runs `label_worker.py` once per GPU, each worker taking every other shard, then checks the
result with `validate_local.py`. Shards go to the store in `storage.backend`: the team's GCS
bucket (`storage.gcs_bucket`), or the private Hugging Face dataset (`storage.hf_repo`), the
backup. Rerunning resumes: finished shards (manifest present) are skipped.

**`RUN` in the settings cell picks the job:**
* `"dryrun"`: two 100-row shards under `teacher/<prompt_version>-dryrun/`. After the first pass
  it deletes one manifest (what a crash between the two uploads leaves), reruns, and checks
  that only that shard is redone. About 20 to 30 minutes, nearly all setup.
* `"train"`: all 159,571 training rows, 32 shards, about 3.6 h on two T4s. Use
  "Save & Run All" so it keeps running without the browser.
* `"gate"`: the 20,000-row gate subset of the test split (`artifacts/gate_subset_ids.parquet`),
  4 shards under `teacher/<prompt_version>/gate/`, about 45 minutes on two T4s.

**Kaggle (preferred):** Accelerator **GPU T4 x2**, Internet **On**, Add Input → Your Work →
Notebooks → **00 prepare data**. Add-ons → Secrets, for the configured backend:
* `gcs`: `GCS_TEACHER_KEY`, a service-account key (the whole JSON) that can write the bucket's
  `teacher/` and `soft_labels/` prefixes;
* `hf`: `HF_TOKEN` with **write** access to the shard dataset.

**Colab:** one T4, so one worker (about twice as long). The same secret in Colab Secrets (key icon),
the train parquet uploaded to `/content` or in Drive, as for the pilot notebook.

In [ ]:
import time

SESSION_START = time.time()   # the labeling time budget counts from here

REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "ab6a30fbc8c8a01e8661a4c581243f40cedbb31c"   # pinned: reruns must use the same code
VLLM_VERSION = "0.31.0"
RUN = "gate"                      # "dryrun", "train", or "gate"
NUM_WORKERS = None                # None: one per GPU

# Colab only (ignored on Kaggle).
COLAB_TRAIN = "/content/wiki_train_raw.parquet"
COLAB_TEST = "/content/wiki_test_raw.parquet"    # for RUN = "gate"

# Testing only: a local folder instead of Hugging Face, and the worker command.
LOCAL_ROOT = None
WORKER_CMD = None

In [ ]:
# Install vLLM before anything imports torch. 3 to 10 minutes; the dependency warnings at the
# end are about packages this notebook does not use.
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", f"vllm=={VLLM_VERSION}"], check=True)

# The image's torchaudio is built for a different CUDA than vLLM's torch, and transformers
# imports it if present. The workers and vLLM's own helper processes start fresh, so hiding it
# in this kernel is not enough: remove it. Nothing here needs audio.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=True)

In [ ]:
from pathlib import Path

REPO_DIR = Path("/tmp/Distillate")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
PIPELINE = REPO_DIR / "Model-Pipeline"
sys.path.insert(0, str(PIPELINE))
print("code_commit:", CODE_COMMIT)

In [ ]:
# GPUs, read with nvidia-smi so this kernel never initializes CUDA (the workers need the memory).
gpus = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total",
     "--format=csv,noheader"], text=True).strip().splitlines()
print("\n".join(gpus))
NUM_WORKERS = NUM_WORKERS or len(gpus)
assert NUM_WORKERS >= 1, "no GPU: check the accelerator"
print("workers:", NUM_WORKERS)

In [ ]:
# Store credentials from Secrets, into the environment the workers inherit. Never printed.
# Whichever of the two exist are loaded; the next cell checks the one the backend needs.
import os

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def read_secret(name):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret(name)
        from google.colab import userdata

        return userdata.get(name)
    except Exception:   # secret not attached
        return None


if LOCAL_ROOT is None:
    for name in ("GCS_TEACHER_KEY", "HF_TOKEN"):
        value = read_secret(name)
        if value:
            os.environ[name] = value

In [ ]:
import json

import pandas as pd

from src.teacher.config import load_config, model_version
from src.teacher.data import find_file
from src.teacher.prompts import prompt_version

cfg = load_config()
P = cfg["prompt"]
PROMPT_VERSION = prompt_version(P["protocol"], P["shots"], P["text_version"])
MODEL_VERSION = model_version(cfg)

BACKEND = cfg["storage"]["backend"]
if LOCAL_ROOT is None and BACKEND == "gcs":
    # Fail now, not after the model load, if the key cannot write to the bucket.
    from src.teacher.storage import GCS_KEY_ENV, gcs_client

    assert os.environ.get(GCS_KEY_ENV), f"attach the {GCS_KEY_ENV} secret"
    bucket = gcs_client().bucket(cfg["storage"]["gcs_bucket"])
    needed = ["storage.objects.create", "storage.objects.delete", "storage.objects.get",
              "storage.objects.list"]
    missing = sorted(set(needed) - set(bucket.test_iam_permissions(needed)))
    assert not missing, f"the key lacks {missing} on gs://{bucket.name}"
    print("GCS bucket:", f"gs://{bucket.name}")
elif LOCAL_ROOT is None:
    # Fail now, not after the model load, if the token cannot write to the shard dataset.
    from huggingface_hub import HfApi

    assert os.environ.get("HF_TOKEN"), "attach the HF_TOKEN secret"
    api = HfApi()
    user = api.whoami()["name"]
    repo = api.repo_info(cfg["storage"]["hf_repo"], repo_type="dataset")
    assert repo.private, "the shard dataset must be private"
    print("Hugging Face user:", user, "| dataset:", repo.id)

if ON_KAGGLE:
    TRAIN_PATH = find_file("/kaggle/input", cfg["data"]["files"]["train"])
    TEST_PATH = find_file("/kaggle/input", cfg["data"]["files"]["test"])
    MODEL_DIR = find_file("/kaggle/input", "weights_manifest.json").parent
    weights_manifest = json.loads((MODEL_DIR / "weights_manifest.json").read_text())
    assert weights_manifest["revision"] == cfg["model"]["revision"], weights_manifest["revision"]
    LOG_DIR = Path("/kaggle/working/logs")
else:   # Colab
    from huggingface_hub import snapshot_download

    TRAIN_PATH, TEST_PATH = Path(COLAB_TRAIN), Path(COLAB_TEST)
    MODEL_DIR = Path(snapshot_download(
        repo_id=cfg["model"]["model_id"], revision=cfg["model"]["revision"],
        local_dir="/content/model", allow_patterns=["*.json", "*.safetensors", "*.txt", "LICENSE"]))
    LOG_DIR = Path("/content/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

JOBS = {
    "dryrun": {"data": TRAIN_PATH, "ids": None, "limit": 200, "shard_size": 100, "plan": None,
               "prefix": f"teacher/{PROMPT_VERSION}-dryrun/"},
    "train": {"data": TRAIN_PATH, "ids": None, "limit": None, "shard_size": None,
              "plan": Path(cfg["paths"]["shard_plan"]),
              "prefix": f"teacher/{PROMPT_VERSION}/"},
    "gate": {"data": TEST_PATH, "ids": PIPELINE / cfg["paths"]["gate_ids"],
             "limit": None, "shard_size": None, "plan": None,
             "prefix": f"teacher/{PROMPT_VERSION}/gate/"},
}
JOB = JOBS[RUN]
if JOB["ids"] is not None:
    assert Path(JOB["ids"]).exists(), f"{JOB['ids']} does not exist yet"
print(RUN, "|", PROMPT_VERSION, "|", JOB["prefix"])

In [ ]:
def job_args(job):
    args = ["--data", str(job["data"]), "--prefix", job["prefix"]]
    for flag in ("ids", "limit", "shard_size", "plan"):
        if job[flag] is not None:
            args += [f"--{flag.replace('_', '-')}", str(job[flag])]
    if LOCAL_ROOT is not None:
        args += ["--local-root", str(LOCAL_ROOT)]
    return args


def run_workers(job, tag):
    """Start one worker per GPU, wait for all, and return their final summaries.

    A failed worker is reported only after the others finish: raising earlier would end a
    committed Kaggle session and kill the workers that are still labeling.
    """
    command = WORKER_CMD or [sys.executable, "-m", "src.teacher.label_worker"]
    procs = []
    for w in range(NUM_WORKERS):
        log = open(LOG_DIR / f"{tag}_worker{w}.log", "w")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(w)}
        args = [*command, "--worker", str(w), "--num-workers", str(NUM_WORKERS),
                "--model-dir", str(MODEL_DIR), "--started-at", str(SESSION_START),
                "--code-commit", CODE_COMMIT, *job_args(job)]
        procs.append((w, subprocess.Popen(args, cwd=PIPELINE, env=env, stdout=log,
                                          stderr=subprocess.STDOUT), log))
    summaries, failed = {}, []
    for w, proc, log in procs:
        code = proc.wait()
        log.close()
        lines = (LOG_DIR / f"{tag}_worker{w}.log").read_text().splitlines()
        events = [json.loads(x) for x in lines if x.startswith("{")]
        done = [e for e in events if e.get("event") == "done"]
        if code != 0 or not done:
            print(f"--- worker {w} exited with {code}; last lines of its log:")
            print("\n".join(lines[-40:]))
            failed.append(w)
            continue
        summaries[w] = done[-1]
        print(f"worker {w}: wrote {done[-1]['written']}, skipped {len(done[-1]['skipped'])}, "
              f"stopped before {done[-1]['stopped_before']}, {done[-1]['seconds']} s scoring")
    if failed:
        raise RuntimeError(f"workers {failed} failed (logs in {LOG_DIR}); finished shards are "
                           "kept, so rerunning the notebook resumes")
    return summaries


def validate(job, partial):
    args = [sys.executable, "-m", "src.teacher.validate_local", "--prefix", job["prefix"],
            "--ids", str(job["ids"] or job["data"])]
    for flag in ("limit", "shard_size", "plan"):
        if job[flag] is not None:
            args += [f"--{flag.replace('_', '-')}", str(job[flag])]
    if LOCAL_ROOT is not None:
        args += ["--local-root", str(LOCAL_ROOT)]
    if partial:
        args.append("--partial")
    result = subprocess.run(args, cwd=PIPELINE, capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())
    return result.returncode == 0

## Run

In [ ]:
start = time.time()
first = run_workers(JOB, f"{RUN}_1")
print(f"pass 1: {time.time() - start:.0f} s")

if RUN == "dryrun":
    assert validate(JOB, partial=False), "dry run: shards failed validation"
    # Simulate a crash between the Parquet and manifest uploads of shard_0001, then resume.
    from src.teacher.storage import get_storage, shard_done, shard_paths

    storage = get_storage(cfg, LOCAL_ROOT)
    storage.delete(shard_paths(JOB["prefix"], "shard_0001")[1])
    assert not shard_done(storage, JOB["prefix"], "shard_0001")
    second = run_workers(JOB, f"{RUN}_2")
    written = sorted(s for summary in second.values() for s in summary["written"])
    skipped = sorted(s for summary in second.values() for s in summary["skipped"])
    assert written == ["shard_0001"] and skipped == ["shard_0000"], (written, skipped)
    assert validate(JOB, partial=False), "dry run: shards failed validation after resume"
    print("Dry run passed: finished shard skipped, interrupted shard redone, validator clean.")
else:
    stopped = any(s["stopped_before"] for s in first.values())
    ok = validate(JOB, partial=stopped)
    print("Validator:", "clean" if ok else "FAILED (see above)")
    if stopped:
        print("Time budget reached; rerun this notebook to label the remaining shards.")
    assert ok